# Assignment 02: Fetching Occurrence Records With pygbif

## BIO597 Spatial Analysis of Biodiversity

This assignment gives you more practice with the workflow from Lab 02.

You will use `pygbif` to search GBIF for several snake species, download a small number of occurrence records, convert those records to `GeoDataFrame` objects, and make simple maps and summaries.

Some cells are partly filled in. You should fill in the missing pieces and run the notebook from top to bottom.

Remember, learning to code is often about learning how to strategically copy, paste, and modify. Look back at Lab 02 when you need a model.


## Species for this assignment

Use these species:

* *Storeria dekayi*
* *Agkistrodon contortrix*
* *Pantherophis guttatus*
* one additional snake species of your choice

For each species, we will ask GBIF for georeferenced records so the results can be mapped.


## 1. Import packages

Import the same packages used in Lab 02.

`species` is used for taxonomic name searches. `occ` is used for occurrence record searches.


In [21]:
from pygbif import species
from pygbif import occurrences as occ

import pandas as pd
import geopandas as gpd

## 2. Search for possible name matches

Use `species.name_suggest()` to search for *Storeria dekayi*.

Fill in the species name. Then inspect the first result.


In [24]:
dekayi_suggestions = species.name_suggest(q="Storeria dekay")
# dekayi_suggestions will have a list of dictionaries
# select the first element of this list here and save it as a new variable called `dekayi_match`
dekayi_match = dekayi_suggestions[0]



## 3. Save the taxon key

Pull the `speciesKey` out of the match result and save it as `dekayi_key`.


In [25]:
dekayi_key = dekayi_match["speciesKey"]
dekayi_key

9056579

## 4. Count records before downloading

Use `occ.count()` to count georeferenced GBIF records for *Storeria dekayi*.

This count tells you how many records GBIF has that match your search, not how many you will download in this assignment.


In [26]:
dekayi_count = occ.count(
    taxonKey=dekayi_key,
    isGeoreferenced=True,
)

dekayi_count

49473

## 5. Determine how many _total_ records there are for S. dekayi

## 5. Determine how many _total_ records there are for S. dekayi

The `isGeoreferenced` parameter determines whether occurrences with latlongs are returned.
Make a copy of the call to `occ.count()` as above, but change the `True` to `False`, which
will return only occurrences **without** latlongs. Capture the results in a new variable 
called `dekayi_nolatlong_count` and then add this to `dekayi_count` from the previous cell
to get the total number of records.

In [28]:
# How many total S. Dekayi records are there?
dekayi_nolatlong_count = occ.count(
    taxonKey=dekayi_key,
    isGeoreferenced=False,
)

total_dekayi_records = dekayi_count + dekayi_nolatlong_count
print(total_dekayi_records)


56546


## 6. Fetch up to 100 records

Use `occ.search()` to fetch occurrence records for *Storeria dekayi*.

Keep `limit=100`. Do not request more than 100 records for a species in this assignment.


In [30]:
dekayi_records = occ.search(
    speciesKey=9056579,
    hasCoordinate=True,
    hasGeospatialIssue=False,
    limit=100,
)

dekayi_records.keys()

dict_keys(['offset', 'limit', 'endOfRecords', 'count', 'results', 'facets'])

## 7. Turn the records into a table

The occurrence records are stored in the "results" key of the `dekayi_records` dictionary. Convert that list of records to a pandas DataFrame.


In [35]:
dekayi_df = pd.DataFrame(dekayi_records["results"])
dekayi_df.head()

,key,datasetKey,publishingOrgKey,datasetCategory,installationKey,hostingOrganizationKey,publishingCountry,protocol,lastCrawled,lastParsed,...,eventTime,identificationID,occurrenceRemarks,informationWithheld,projectId,dynamicProperties,vitality,lifeStage,identificationRemarks,sex
0,5938064620,50c9509d-22c7-4a22-a47d-8c48425ef4a7,28eb1a3f-1c15-4a95-931a-4af90ecb574d,[CitizenScience],997448a8-f762-11e1-a439-00145eb45e9a,28eb1a3f-1c15-4a95-931a-4af90ecb574d,US,DWC_ARCHIVE,2026-09-10T19:15:51.157+00:00,2026-09-11T05:22:11.912+00:00,...,16:48:33-06:00,745332273,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,5938201753,50c9509d-22c7-4a22-a47d-8c48425ef4a7,28eb1a3f-1c15-4a95-931a-4af90ecb574d,[CitizenScience],997448a8-f762-11e1-a439-00145eb45e9a,28eb1a3f-1c15-4a95-931a-4af90ecb574d,US,DWC_ARCHIVE,2026-09-10T19:15:51.157+00:00,2026-09-11T05:48:03.041+00:00,...,14:41:00-06:00,746093157,Body about 0.5 cm wide.,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5938457152,50c9509d-22c7-4a22-a47d-8c48425ef4a7,28eb1a3f-1c15-4a95-931a-4af90ecb574d,[CitizenScience],997448a8-f762-11e1-a439-00145eb45e9a,28eb1a3f-1c15-4a95-931a-4af90ecb574d,US,DWC_ARCHIVE,2026-09-10T19:15:51.157+00:00,2026-09-11T05:36:08.412+00:00,...,04:52:51-06:00,745694353,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,5938490464,50c9509d-22c7-4a22-a47d-8c48425ef4a7,28eb1a3f-1c15-4a95-931a-4af90ecb574d,[CitizenScience],997448a8-f762-11e1-a439-00145eb45e9a,28eb1a3f-1c15-4a95-931a-4af90ecb574d,US,DWC_ARCHIVE,2026-09-10T19:15:51.157+00:00,2026-09-11T05:51:42.840+00:00,...,14:00:00-05:00,746922821,First snake of the year !!!,Coordinate uncertainty increased to 28734m at ...,NaN,NaN,NaN,NaN,NaN,NaN
4,5938606080,50c9509d-22c7-4a22-a47d-8c48425ef4a7,28eb1a3f-1c15-4a95-931a-4af90ecb574d,[CitizenScience],997448a8-f762-11e1-a439-00145eb45e9a,28eb1a3f-1c15-4a95-931a-4af90ecb574d,US,DWC_ARCHIVE,2026-09-10T19:15:51.157+00:00,2026-09-11T05:22:15.726+00:00,...,11:27:14-06:00,745678895,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 8. Keep a small set of useful columns

Keep the columns needed for a map and a few simple summaries.

Fill in the latitude column name.


In [36]:
dekayi_small = dekayi_df[
    ["key", "scientificName", "decimalLongitude", "decimalLatitude", "year", "basisOfRecord", "countryCode"]
]

dekayi_small.head()

,key,scientificName,decimalLongitude,decimalLatitude,year,basisOfRecord,countryCode
0,5938064620,"Storeria dekayi (Holbrook, 1839)",-97.122430,33.241408,2026,HUMAN_OBSERVATION,US
1,5938201753,"Storeria dekayi (Holbrook, 1839)",-92.908814,34.619900,2026,HUMAN_OBSERVATION,US
2,5938457152,"Storeria dekayi (Holbrook, 1839)",-86.721496,33.458035,2026,HUMAN_OBSERVATION,US
3,5938490464,"Storeria dekayi (Holbrook, 1839)",-80.744301,35.172939,2026,HUMAN_OBSERVATION,US
4,5938606080,"Storeria dekayi (Holbrook, 1839)",-97.508688,35.238997,2026,HUMAN_OBSERVATION,US


## 9. Convert the table to a GeoDataFrame

Use the longitude and latitude columns to create point geometry. This is the same idea as Lab 01 and Lab 02.


In [37]:
dekayi_gdf = gpd.GeoDataFrame(
    dekayi_small,
    geometry=gpd.points_from_xy(dekayi_small["decimalLongitude"], dekayi_small["decimalLatitude"]),
    crs="EPSG:4326",
)

dekayi_gdf["Species"] = "Storeria dekayi"
dekayi_gdf.head()

,key,scientificName,decimalLongitude,decimalLatitude,year,basisOfRecord,countryCode,geometry,Species
0,5938064620,"Storeria dekayi (Holbrook, 1839)",-97.122430,33.241408,2026,HUMAN_OBSERVATION,US,POINT (-97.12243 33.24141),Storeria dekayi
1,5938201753,"Storeria dekayi (Holbrook, 1839)",-92.908814,34.619900,2026,HUMAN_OBSERVATION,US,POINT (-92.90881 34.6199),Storeria dekayi
2,5938457152,"Storeria dekayi (Holbrook, 1839)",-86.721496,33.458035,2026,HUMAN_OBSERVATION,US,POINT (-86.7215 33.45804),Storeria dekayi
3,5938490464,"Storeria dekayi (Holbrook, 1839)",-80.744301,35.172939,2026,HUMAN_OBSERVATION,US,POINT (-80.7443 35.17294),Storeria dekayi
4,5938606080,"Storeria dekayi (Holbrook, 1839)",-97.508688,35.238997,2026,HUMAN_OBSERVATION,US,POINT (-97.50869 35.239),Storeria dekayi


## 10. Map *Storeria dekayi*

Make an interactive map of the *Storeria dekayi* records.


In [40]:
dekayi_gdf.explore()

## 11. Repeat the workflow for *Agkistrodon contortrix*

Now repeat the same steps for eastern copperhead, *Agkistrodon contortrix*.

This cell should get the name suggestions and save the speciesKey.


In [41]:
contortrix_suggestions = species.name_suggest(q="Agkistrodon contortrix")

# Select the first element from the `contortrix_suggestions` list
contortrix_match = contortrix_suggestions[0]

# Get the `speciesKey`
contortrix_key = contortrix_match["speciesKey"]

contortrix_key

9215881

## 13. Fetch and map *Agkistrodon contortrix*

Write code to fetch up to 100 records with coordinates, convert them to a table, convert that table to a GeoDataFrame, add a `Species` column, and map the result.

Use the same variable names shown in the comments. In `occ.search()`, use `hasCoordinate=True` and `hasGeospatialIssue=False`.


In [50]:
# Create contortrix_records with occ.search().
contortrix_records = occ.search(
    taxonKey=9215881,
    hasCoordinate=True,
    isGeoreferenced=False,
    limit=100,
)

# Create contortrix_df from contortrix_records["results"].

contortrix_df = pd.DataFrame(contortrix_records["results"])
contortrix_df.head()

# Create contortrix_small with the columns you want to keep.

contortrix_small = contortrix_df[
    ["key", "scientificName", "decimalLongitude", "decimalLatitude", "year", "basisOfRecord", "countryCode"]
]

contortrix_small.head()


# Create contortrix_gdf with gpd.GeoDataFrame().

contortrix_gdf = gpd.GeoDataFrame(
    contortrix_small,
    geometry=gpd.points_from_xy(contortrix_small["decimalLongitude"], contortrix_small["decimalLatitude"]),
    crs="EPSG:4326",
)

# Add a Species column with the name Agkistrodon contortrix.

contortrix_gdf["Species"] = "Agkistrodon contortrix"
contortrix_gdf.head()

# Map contortrix_gdf with .explore().

contortrix_gdf.explore()


## 14. Repeat the workflow for *Pantherophis guttatus*

This time you will do a little more on your own.

First, use `species.name_suggest()` to get the `speciesKey` for *Pantherophis guttatus*.


In [52]:
# Write your code here.
pguttatus_suggestions = species.name_suggest(q="Pantherophis guttatus")

pguttatus_match = pguttatus_suggestions[0]
pguttatus_key = pguttatus_match["speciesKey"]
pguttatus_key


2455615

## 16. Fetch and map *Pantherophis guttatus*

Fetch up to 100 georeferenced records and convert them to a GeoDataFrame.

Keep the cell simple. It is fine to copy and modify code from earlier cells.


In [60]:
# Write your code here.
pguttatus_records = occ.search(
    speciesKey=2455615,
    hasCoordinate=True,
    isGeoreferenced=False,
    limit=100,
)
pguttatus_records.keys()

pguttatus_df = pd.DataFrame(pguttatus_records["results"])
pguttatus_df.head()

pguttatus_small = pguttatus_df[
    ["key", "scientificName", "decimalLongitude", "decimalLatitude", "year", "countryCode"]
]

pguttatus_small.head()

pguttatus_gdf = gpd.GeoDataFrame(
    pguttatus_small,
    geometry=gpd.points_from_xy(pguttatus_small["decimalLongitude"], pguttatus_small["decimalLatitude"]),
    crs="EPSG:4326",
)

pguttatus_gdf["Species"] = "Pantherophis guttatus"
pguttatus_gdf.head()

pguttatus_gdf.explore()


## 17. Combine the three GeoDataFrames

Use `pd.concat()` to combine your three species GeoDataFrames.

Then inspect the first few rows.


In [61]:
gbif_snakes = pd.concat([pguttatus_gdf, contortrix_gdf, dekayi_gdf])

gbif_snakes.head()

,key,scientificName,decimalLongitude,decimalLatitude,year,countryCode,geometry,Species,basisOfRecord
0,5938135213,"Pantherophis guttatus (Linnaeus, 1766)",-81.394973,30.135123,2026,US,POINT (-81.39497 30.13512),Pantherophis guttatus,NaN
1,5938250009,"Pantherophis guttatus (Linnaeus, 1766)",-80.278505,25.329077,2026,US,POINT (-80.2785 25.32908),Pantherophis guttatus,NaN
2,5938256663,"Pantherophis guttatus (Linnaeus, 1766)",-80.452592,27.789722,2026,US,POINT (-80.45259 27.78972),Pantherophis guttatus,NaN
3,5938387083,"Pantherophis guttatus (Linnaeus, 1766)",-88.015381,30.231975,2026,US,POINT (-88.01538 30.23198),Pantherophis guttatus,NaN
4,5938619022,"Pantherophis guttatus (Linnaeus, 1766)",-81.797488,26.245595,2026,US,POINT (-81.79749 26.2456),Pantherophis guttatus,NaN


## 18. Map all three species together

Use `.explore()` and color by `Species` so you can compare the three species on one map.


In [62]:
gbif_snakes.explore(column="Species", cmap="rainbow")

## 19. Count records by species

Use `groupby()` to count how many records you downloaded for each species.


In [64]:
gbif_snakes.groupby("Species").size()

Species
Agkistrodon contortrix    100
Pantherophis guttatus     100
Storeria dekayi           100
dtype: int64

## 20. Count records by basis of record

The `countryCode` field describes the general type of occurrence record.

Use `groupby()` to count records by `countryCode`.


In [65]:
gbif_snakes.groupby("countryCode").size()

countryCode
MX      2
US    298
dtype: int64

## 21. Choose one additional species

Choose one additional snake species and repeat the workflow.

Your species does not have to be in the local `EasternSnakes` CSV files. It only needs to be a snake species that GBIF can find.

Your code should:

* use `species.name_suggest()`
* save the taxon key
* count georeferenced records
* fetch no more than 100 records with `occ.search()`
* convert the records to a `GeoDataFrame`
* map the records


In [79]:
# Write your code here.
# Thamnophis sirtalis, Common Garter Snake

tsirtalis_suggestions = species.name_suggest(q="Thamnophis sirtalis")
tsirtalis_match = tsirtalis_suggestions[0]

tsirtalis_key = tsirtalis_match["speciesKey"]
tsirtalis_key
# 2457522

tsirtalis_records = occ.search(
    speciesKey=2457522,
    hasCoordinate=True,
    hasGeospatialIssue=False,
    limit=100,
)

tsirtalis_records.keys()

tsirtalis_df =pd.DataFrame(tsirtalis_records["results"])
tsirtalis_df.head()

tsirtalis_small = tsirtalis_df[
    ["key", "scientificName", "decimalLongitude", "decimalLatitude", "year", "countryCode"]
]

tsirtalis_small.head()

tsirtalis_gdf = gpd.GeoDataFrame(
    tsirtalis_small,
    geometry=gpd.points_from_xy(tsirtalis_small["decimalLongitude"], tsirtalis_small["decimalLatitude"]),
    crs="EPSG:4326",
)

tsirtalis_gdf["Species"] = "Thamnophis sirtalis"
tsirtalis_gdf.head()

tsirtalis_gdf.explore()

## 22. Written reflection

Answer these questions after running your code.

**Question 1:** Which of your species had the most GBIF records available?

**Your answer:**  Storeria dekayi has the most GBIF records available. (counts shown below)

**Question 2:** Did the GBIF points look similar to the local CSV points from Assignment 01? Why might GBIF records look different?

**Your answer:** No, the GBIF points did not look similar to the local CSV points. The CSV points from Assignment 01 were from specific field studies whereas GBIF is more likely getting its data from citizen science projects. 

**Question 3:** What is one reason it is useful to count records before downloading or mapping them?

**Your answer:** So you do not download more than you need


In [81]:
dekayi_count = occ.count(
    taxonKey=dekayi_key,
    isGeoreferenced=True,
)
dekayi_count

dekayi_nolatlong_count = occ.count(
    taxonKey=dekayi_key,
    isGeoreferenced=False,
)
total_dekayi_records = dekayi_count + dekayi_nolatlong_count
print(total_dekayi_records)

56546


In [86]:
contortrix_count = occ.count(
    taxonKey=contortrix_key,
    isGeoreferenced=True,
)
contortrix_count

contortrix_nolatlong_count = occ.count(
    taxonKey=contortrix_key,
    isGeoreferenced=False,
)
total_contortrix_records = contortrix_count + contortrix_nolatlong_count
print(total_contortrix_records)

29115


In [87]:
pguttatus_count = occ.count(
    taxonKey=contortrix_key,
    isGeoreferenced=True,
)
pguttatus_count

pguttatus_nolatlong_count = occ.count(
    taxonKey=pguttatus_key,
    isGeoreferenced=False,
)
total_pguttatus_records = pguttatus_count + pguttatus_nolatlong_count
print(total_pguttatus_records)

26368


In [88]:
tsirtalis_count = occ.count(
    taxonKey=contortrix_key,
    isGeoreferenced=True,
)
tsirtalis_count

tsirtalis_nolatlong_count = occ.count(
    taxonKey=tsirtalis_key,
    isGeoreferenced=False,
)
total_tsirtalis_records = tsirtalis_count + tsirtalis_nolatlong_count
print(total_tsirtalis_records)

45320


## 23. Submit your work

Before submitting, make sure you have run the notebook from top to bottom and answered the written questions.

Commit and push your completed notebook to your class GitHub repository.

Open a terminal window and run these commands to add, commit, and push your notebook:

```
# Go to the labs directory in the course repo
cd ~/BIO597-SpatialBiodiversity/docs/assignments

# Add your changed lab
git add Assignment-02-pygbif.ipynb
git commit -m 'Finished Assignment 02'
git push
```